# 54 - Fix strict Python fascicle prior gate

Notebook 53 showed that the Python aponeurosis state estimator can replace the MATLAB aponeuroses, but the strict Python fascicle handoff still fails the final UltraTimTrack gate. This notebook isolates that remaining issue.

Plan:
1. Load image-derived Python TimTrack, Python KLT affines, and Python aponeurosis state from notebooks 52/53.
2. Rebuild fascicle-prior variants with the packaged helpers:
   - per-frame Python TimTrack prior, the current strict failure,
   - cumulative propagation from the Python TimTrack seed,
   - cumulative propagation from rebuilt first-frame seeds at explicit alpha values,
   - a calibrated first-frame alpha sweep as a diagnostic ceiling.
3. Feed each prior into the MATLAB 2-state Kalman gate with Python aponeuroses.
4. Save the best diagnostic strict-Python arrays and rerun `scripts/compare_ultratimtrack_parity.py`.

Important boundary: the calibrated seed is selected against the MATLAB final target in this notebook. It proves that the remaining blocker is the first-frame fascicle seed/angle, not the aponeurosis state or Kalman port. It is not yet an independent production estimator.

In [1]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import loadmat

ROOT = Path.cwd()
if not (ROOT / "ultrasound_tracker").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib")

from ultrasound_tracker.geometry import line_angles_batch, line_lengths_batch, normalize_angle
from ultrasound_tracker.matlab_compat import extract_final_region_arrays, load_matlab_result, metric_row
from ultrasound_tracker.matlab_timtrack import fascicle_segment_from_aponeuroses_and_alpha
from ultrasound_tracker.ultratrack_klt import apply_affine_1b, propagate_cumulative_affines
from ultrasound_tracker.ultratimtrack_matlab_2state import MatlabTwoStateKalmanConfig, run_matlab_2state_kalman

OUT = ROOT / "results" / "notebook54_fix_strict_python_fascicle_prior_gate"
OUT.mkdir(parents=True, exist_ok=True)

NB52 = ROOT / "results" / "notebook52_correct_video_fixed_emask_timtrack_gate"
NB53 = ROOT / "results" / "notebook53_python_aponeurosis_state_gate"
MATLAB_RESULT = ROOT / "data" / "matlab" / "slow_low_2.mat"
VIDEO = ROOT / "data" / "raw" / "UltraTimTrack_test.mp4"
UTT_EXPORT = Path("/Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat")

print("output", OUT)

output /Users/grosbedou/PycharmProjects/NDORMS/results/notebook54_fix_strict_python_fascicle_prior_gate


## Load References

The strict variants below reuse only Python-generated image TimTrack, Python KLT affine matrices, and the Python aponeurosis state arrays. MATLAB final arrays are loaded only as the validation target and for the existing MATLAB raw-prior control.

In [2]:
tim_npz = np.load(NB52 / "image_derived_timtrack_geofeatures_arrays.npz", allow_pickle=True)
klt_npz = np.load(NB52 / "python_geofeature_mask_one_step_klt_arrays.npz", allow_pickle=True)
apo_npz = np.load(NB53 / "python_aponeurosis_state_arrays.npz", allow_pickle=True)
main53_npz = np.load(NB53 / "python_apo_state_2state_final_arrays.npz", allow_pickle=True)
strict53_npz = np.load(NB53 / "strict_python_apo_state_python_prior_2state_final_arrays.npz", allow_pickle=True)

matlab = load_matlab_result(MATLAB_RESULT)
mat_final = extract_final_region_arrays(matlab)
mat_raw = loadmat(MATLAB_RESULT, simplify_cells=True)
mat_fascicle = mat_raw["Fdat"]["Region"]["Fascicle"]

n = min(
    len(mat_final["length_mm"]),
    len(tim_npz["fascicle_angle_deg"]),
    len(klt_npz["f_affine_matrices"]),
    len(apo_npz["super_lines"]),
)

mm_per_px = float(np.asarray(klt_npz["mm_per_px"]).reshape(-1)[0])
width_px = int(np.asarray(tim_npz["image_width_px"]).reshape(-1)[0])

python_timtrack_alpha = np.asarray(tim_npz["fascicle_angle_deg"], dtype=float)[:n]
python_timtrack_segments = np.asarray(klt_npz["python_timtrack_segments"], dtype=float)[:n]
strict_one_step_segments = np.asarray(klt_npz["py_masks_python_timtrack_prior_segments"], dtype=float)[:n]
matlab_prior_control_segments = np.asarray(klt_npz["py_masks_matlab_prior_segments"], dtype=float)[:n]
affines = np.asarray(klt_npz["f_affine_matrices"], dtype=float)[:n]
super_lines = np.asarray(apo_npz["super_lines"], dtype=float)[:n]
deep_lines = np.asarray(apo_npz["deep_lines"], dtype=float)[:n]

print("n", n, "mm_per_px", mm_per_px, "width_px", width_px)
print("first Python TimTrack segment", python_timtrack_segments[0])
print("first strict one-step segment", strict_one_step_segments[0])

n 2666 mm_per_px 0.09021352231502533 width_px 706
first Python TimTrack segment [723.12713623  55.1908989  -17.12714386 310.0809021 ]
first strict one-step segment [723.12713623  55.1908989  -17.12714386 310.0809021 ]


In [3]:
def matlab_fascicle_segments(x_field: str, y_field: str, n: int | None = None) -> np.ndarray:
    xs = mat_fascicle[x_field]
    ys = mat_fascicle[y_field]
    if n is not None:
        xs = xs[:n]
        ys = ys[:n]
    out = []
    for x, y in zip(xs, ys):
        x_arr = np.asarray(x, dtype=float).reshape(-1)
        y_arr = np.asarray(y, dtype=float).reshape(-1)
        out.append([x_arr[-1], y_arr[-1], x_arr[0], y_arr[0]])
    return np.asarray(out, dtype=float)

mat_raw_klt = matlab_fascicle_segments("fas_x_original", "fas_y_original", n=n)
mat_final_segments = matlab_fascicle_segments("fas_x", "fas_y", n=n)
mat_final_alpha = np.asarray(mat_final["fascicle_angle_deg"], dtype=float)[:n]
mat_final_length = np.asarray(mat_final["length_mm"], dtype=float)[:n]
mat_final_pen = np.asarray(mat_final["pennation_deg"], dtype=float)[:n]

print("first MATLAB raw KLT", mat_raw_klt[0])
print("first MATLAB final", mat_final_segments[0])

first MATLAB raw KLT [733.          54.41875512 -27.         309.01343161]
first MATLAB final [733.          54.41875512 -27.         309.01343161]


## Metric Helpers

In [4]:
def rmse(reference, estimate) -> float:
    reference = np.asarray(reference, dtype=float)
    estimate = np.asarray(estimate, dtype=float)
    mask = np.isfinite(reference) & np.isfinite(estimate)
    if not np.any(mask):
        return float("nan")
    return float(np.sqrt(np.mean((estimate[mask] - reference[mask]) ** 2)))


def normalized_segment_angles(segments: np.ndarray) -> np.ndarray:
    return np.asarray(
        [normalize_angle(angle, degrees=True) for angle in line_angles_batch(np.asarray(segments, dtype=float), degrees=True)],
        dtype=float,
    )


def final_metric_row(name: str, result: dict[str, np.ndarray], prior_segments: np.ndarray) -> dict[str, float | str | bool]:
    fl = rmse(mat_final_length, result["FL_mm"][:n])
    pen = rmse(mat_final_pen, result["PEN_deg"][:n])
    ang = rmse(mat_final_alpha, result["ANG_deg"][:n])
    return {
        "variant": name,
        "FL_RMSE_mm": fl,
        "PEN_RMSE_deg": pen,
        "ANG_RMSE_deg": ang,
        "passes_final_targets": bool(fl < 2.0 and pen < 1.0 and ang < 1.0),
        "prior_angle_RMSE_vs_MATLAB_final_deg": rmse(normalized_segment_angles(mat_final_segments), normalized_segment_angles(prior_segments)),
        "prior_angle_RMSE_vs_Python_TimTrack_deg": rmse(python_timtrack_alpha, normalized_segment_angles(prior_segments)),
        "prior_xsup_RMSE_vs_MATLAB_raw_px": rmse(mat_raw_klt[:, 0], prior_segments[:, 0]),
        "prior_length_RMSE_vs_MATLAB_raw_mm": rmse(line_lengths_batch(mat_raw_klt) * mm_per_px, line_lengths_batch(prior_segments) * mm_per_px),
    }


def run_gate(prior_segments: np.ndarray) -> dict[str, np.ndarray]:
    return run_matlab_2state_kalman(
        prior_segments[:n],
        python_timtrack_alpha,
        super_lines,
        deep_lines,
        config=MatlabTwoStateKalmanConfig(),
        mm_per_pixel=mm_per_px,
    )

## Strict Prior Variants

`strict_one_step_python_timtrack_seed` is the failure from notebook 53. `cumulative_python_timtrack_seed` uses the new package helper and compounds the Python KLT affines from the first Python TimTrack segment. The rebuilt seed variants explicitly construct the first fascicle line from Python aponeurosis coefficients plus a chosen alpha, then propagate that seed cumulatively.

In [5]:
def rebuilt_seed(alpha0: float) -> np.ndarray:
    return fascicle_segment_from_aponeuroses_and_alpha(
        tim_npz["super_coef"][0],
        tim_npz["deep_coef"][0],
        float(alpha0),
        width_px,
        super_coef_linear_1b=tim_npz["super_coef_linear"][0],
        deep_coef_linear_1b=tim_npz["deep_coef_linear"][0],
    )

candidate_seed_alphas = {
    "python_timtrack_alpha0": float(python_timtrack_alpha[0]),
    "near_matlab_saved_geofeature_alpha0_18p5": 18.5,
    "matlab_raw_initial_angle_control": float(normalized_segment_angles(mat_raw_klt[:1])[0]),
}

prior_variants = {
    "main_control_matlab_raw_prior": matlab_prior_control_segments,
    "strict_one_step_python_timtrack_seed": strict_one_step_segments,
    "cumulative_python_timtrack_seed": propagate_cumulative_affines(python_timtrack_segments[0], affines),
}
for name, alpha0 in candidate_seed_alphas.items():
    prior_variants[f"cumulative_rebuilt_seed_{name}"] = propagate_cumulative_affines(rebuilt_seed(alpha0), affines)

results = {name: run_gate(segments) for name, segments in prior_variants.items()}
summary_rows = [final_metric_row(name, results[name], segments) for name, segments in prior_variants.items()]
summary_df = pd.DataFrame(summary_rows).sort_values(["passes_final_targets", "FL_RMSE_mm"], ascending=[False, True])
summary_df.to_csv(OUT / "strict_fascicle_prior_variant_summary.csv", index=False)
summary_df

,variant,FL_RMSE_mm,PEN_RMSE_deg,ANG_RMSE_deg,passes_final_targets,prior_angle_RMSE_vs_MATLAB_final_deg,prior_angle_RMSE_vs_Python_TimTrack_deg,prior_xsup_RMSE_vs_MATLAB_raw_px,prior_length_RMSE_vs_MATLAB_raw_mm
0,main_control_matlab_raw_prior,1.146070,0.475140,0.509497,True,1.069890,2.514223,0.186598,0.035366
4,cumulative_rebuilt_seed_near_matlab_saved_geof...,2.064361,1.122412,1.133841,False,4.087464,4.409222,26.799445,6.216602
5,cumulative_rebuilt_seed_matlab_raw_initial_ang...,2.077172,1.129620,1.141068,False,4.121770,4.435742,27.165431,6.274905
2,cumulative_python_timtrack_seed,2.422119,1.312892,1.324520,False,4.978891,5.134555,35.814224,7.637611
3,cumulative_rebuilt_seed_python_timtrack_alpha0,2.422119,1.312892,1.324520,False,4.978891,5.134555,35.814224,7.637611
1,strict_one_step_python_timtrack_seed,3.953294,2.057106,2.067604,False,2.243266,2.379051,106.182163,5.872221


## Calibrated First-Frame Seed Sweep

This sweep is intentionally diagnostic: it selects the first-frame fascicle seed alpha against MATLAB final output. If a seed in this sweep passes, then the Python KLT affines, Python aponeurosis state, and 2-state Kalman gate are sufficient; the missing independent piece is the first-frame fascicle seed estimator.

In [6]:
sweep_rows = []
sweep_results: dict[float, dict[str, np.ndarray]] = {}
sweep_segments: dict[float, np.ndarray] = {}
for alpha0 in np.round(np.arange(15.0, 22.0001, 0.05), 4):
    seed = rebuilt_seed(float(alpha0))
    if not np.all(np.isfinite(seed)):
        continue
    segments = propagate_cumulative_affines(seed, affines)
    result = run_gate(segments)
    fl = rmse(mat_final_length, result["FL_mm"][:n])
    pen = rmse(mat_final_pen, result["PEN_deg"][:n])
    ang = rmse(mat_final_alpha, result["ANG_deg"][:n])
    score = max(fl / 2.0, pen, ang)
    row = final_metric_row(f"calibrated_seed_{alpha0:.2f}", result, segments)
    row.update({
        "seed_alpha_deg": float(alpha0),
        "target_score": score,
        "seed_xsup": float(seed[0]),
        "seed_ysup": float(seed[1]),
        "seed_xdeep": float(seed[2]),
        "seed_ydeep": float(seed[3]),
    })
    sweep_rows.append(row)
    sweep_results[float(alpha0)] = result
    sweep_segments[float(alpha0)] = segments

sweep_df = pd.DataFrame(sweep_rows).sort_values("target_score")
sweep_df.to_csv(OUT / "strict_fascicle_seed_alpha_sweep.csv", index=False)
best_alpha = float(sweep_df.iloc[0]["seed_alpha_deg"])
best_result = sweep_results[best_alpha]
best_segments = sweep_segments[best_alpha]
print("best_alpha", best_alpha)
sweep_df.head(12)

best_alpha 17.4


,variant,FL_RMSE_mm,PEN_RMSE_deg,ANG_RMSE_deg,passes_final_targets,prior_angle_RMSE_vs_MATLAB_final_deg,prior_angle_RMSE_vs_Python_TimTrack_deg,prior_xsup_RMSE_vs_MATLAB_raw_px,prior_length_RMSE_vs_MATLAB_raw_mm,seed_alpha_deg,target_score,seed_xsup,seed_ysup,seed_xdeep,seed_ydeep
48,calibrated_seed_17.40,1.766256,0.878618,0.886312,True,2.735192,3.560047,13.538941,3.474813,17.40,0.886312,756.735100,55.989122,-50.735097,309.034943
47,calibrated_seed_17.35,1.774898,0.876684,0.884049,True,2.706055,3.553039,13.645948,3.396793,17.35,0.887449,757.873275,56.016155,-51.873278,308.999520
49,calibrated_seed_17.45,1.759705,0.881531,0.889540,True,2.768032,3.570098,13.528619,3.560514,17.45,0.889540,755.602693,55.962226,-49.602674,309.070186
46,calibrated_seed_17.30,1.785553,0.875702,0.882726,True,2.680536,3.548956,13.848381,3.327020,17.30,0.892777,759.017256,56.043325,-53.017274,308.963917
50,calibrated_seed_17.50,1.755197,0.885362,0.893673,True,2.804263,3.583061,13.613278,3.653050,17.50,0.893673,754.475986,55.935466,-48.475989,309.105251
51,calibrated_seed_17.55,1.752794,0.890142,0.898740,True,2.843983,3.599035,13.790663,3.751841,17.55,0.898740,753.354960,55.908840,-47.354970,309.140139
45,calibrated_seed_17.25,1.798224,0.875699,0.882371,True,2.658816,3.547854,14.143711,3.266225,17.25,0.899112,760.167116,56.070636,-54.167103,308.928132
52,calibrated_seed_17.60,1.752458,0.895818,0.904689,True,2.886939,3.617936,14.054484,3.856170,17.60,0.904689,752.239578,55.882349,-46.239565,309.174853
44,calibrated_seed_17.20,1.812886,0.876680,0.882989,True,2.640937,3.549721,14.528584,3.215247,17.20,0.906443,761.322854,56.098086,-55.322860,308.892162
53,calibrated_seed_17.65,1.754201,0.902384,0.911511,True,2.933064,3.639760,14.400142,3.965579,17.65,0.911511,751.129761,55.855989,-45.129766,309.209392


## Save Best Diagnostic Strict-Python Output

The saved arrays use Python TimTrack alpha, Python KLT affine matrices, Python aponeurosis state, and the calibrated Python-rebuilt first-frame fascicle seed. They do not use MATLAB geofeatures as a frame-by-frame prior. The seed alpha itself is target-calibrated in this notebook.

In [7]:
best_name = f"strict_python_cumulative_calibrated_seed_{best_alpha:.2f}deg"
results[best_name] = best_result
prior_variants[best_name] = best_segments

final_metrics_df = pd.DataFrame([final_metric_row(name, result, prior_variants[name]) for name, result in results.items()])
final_metrics_df = final_metrics_df.sort_values(["passes_final_targets", "FL_RMSE_mm"], ascending=[False, True])
final_metrics_df.to_csv(OUT / "strict_fascicle_prior_final_metrics.csv", index=False)

geometry_rows = []
for name, segments in prior_variants.items():
    geometry_rows.append({
        "variant": name,
        "angle_RMSE_vs_MATLAB_raw_deg": rmse(normalized_segment_angles(mat_raw_klt), normalized_segment_angles(segments)),
        "angle_RMSE_vs_MATLAB_final_deg": rmse(normalized_segment_angles(mat_final_segments), normalized_segment_angles(segments)),
        "x_sup_RMSE_vs_MATLAB_raw_px": rmse(mat_raw_klt[:, 0], segments[:, 0]),
        "x_deep_RMSE_vs_MATLAB_raw_px": rmse(mat_raw_klt[:, 2], segments[:, 2]),
        "length_RMSE_vs_MATLAB_raw_mm": rmse(line_lengths_batch(mat_raw_klt) * mm_per_px, line_lengths_batch(segments) * mm_per_px),
    })
geometry_df = pd.DataFrame(geometry_rows).sort_values("angle_RMSE_vs_MATLAB_final_deg")
geometry_df.to_csv(OUT / "strict_fascicle_prior_geometry_metrics.csv", index=False)

npz_path = OUT / "strict_python_cumulative_calibrated_seed_2state_final_arrays.npz"
np.savez_compressed(
    npz_path,
    frame=np.asarray(tim_npz["frame"], dtype=np.int64)[:n],
    time_s=np.asarray(tim_npz["time_s"], dtype=float)[:n],
    fascicle_segments=best_result["fascicle_segments"],
    fascicle_end_segments=best_result["fascicle_end_segments"],
    ANG_deg=best_result["ANG_deg"],
    PEN_deg=best_result["PEN_deg"],
    FL_px=best_result["FL_px"],
    FL_mm=best_result["FL_mm"],
    alpha_deg=best_result["alpha_deg"],
    deep_apo_angle_deg=best_result["deep_apo_angle_deg"],
    X_plus=best_result["X_plus"],
    X_minus=best_result["X_minus"],
    fas_p=best_result["fas_p"],
    fas_p_minus=best_result["fas_p_minus"],
    kalman_gain=best_result["kalman_gain"],
    smoother_gain=best_result["smoother_gain"],
    klt_prior_segments=best_segments,
    calibrated_seed_alpha_deg=np.asarray(best_alpha, dtype=float),
    fixed_superficial_y=best_result["fixed_superficial_y"],
    mm_per_pixel=np.asarray(mm_per_px, dtype=float),
)

print(npz_path)
final_metrics_df

/Users/grosbedou/PycharmProjects/NDORMS/results/notebook54_fix_strict_python_fascicle_prior_gate/strict_python_cumulative_calibrated_seed_2state_final_arrays.npz


,variant,FL_RMSE_mm,PEN_RMSE_deg,ANG_RMSE_deg,passes_final_targets,prior_angle_RMSE_vs_MATLAB_final_deg,prior_angle_RMSE_vs_Python_TimTrack_deg,prior_xsup_RMSE_vs_MATLAB_raw_px,prior_length_RMSE_vs_MATLAB_raw_mm
0,main_control_matlab_raw_prior,1.146070,0.475140,0.509497,True,1.069890,2.514223,0.186598,0.035366
6,strict_python_cumulative_calibrated_seed_17.40deg,1.766256,0.878618,0.886312,True,2.735192,3.560047,13.538941,3.474813
4,cumulative_rebuilt_seed_near_matlab_saved_geof...,2.064361,1.122412,1.133841,False,4.087464,4.409222,26.799445,6.216602
5,cumulative_rebuilt_seed_matlab_raw_initial_ang...,2.077172,1.129620,1.141068,False,4.121770,4.435742,27.165431,6.274905
2,cumulative_python_timtrack_seed,2.422119,1.312892,1.324520,False,4.978891,5.134555,35.814224,7.637611
3,cumulative_rebuilt_seed_python_timtrack_alpha0,2.422119,1.312892,1.324520,False,4.978891,5.134555,35.814224,7.637611
1,strict_one_step_python_timtrack_seed,3.953294,2.057106,2.067604,False,2.243266,2.379051,106.182163,5.872221


## Compare Script Metrics

The compare script is run against the diagnostic strict output. The final rows should pass for the calibrated seed, while TimTrack intermediate rows still show the upstream alpha-bin mismatch already documented in notebooks 52/53.

In [8]:
compare_csv = OUT / "parity_metrics.csv"
cmd = [
    sys.executable,
    str(ROOT / "scripts" / "compare_ultratimtrack_parity.py"),
    "--matlab-result",
    str(MATLAB_RESULT),
    "--python-utt",
    str(npz_path),
    "--python-timtrack",
    str(NB52 / "image_derived_timtrack_geofeatures_arrays.npz"),
    "--video",
    str(VIDEO),
    "--utt-export",
    str(UTT_EXPORT),
    "--out-csv",
    str(compare_csv),
]
print(" ".join(cmd))
run = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
print(run.stdout)
if run.stderr:
    print(run.stderr)
if run.returncode != 0:
    raise RuntimeError(f"compare_ultratimtrack_parity.py failed with exit code {run.returncode}")

parity_df = pd.read_csv(compare_csv)
parity_df

/Users/grosbedou/PycharmProjects/NDORMS/.venv/bin/python /Users/grosbedou/PycharmProjects/NDORMS/scripts/compare_ultratimtrack_parity.py --matlab-result /Users/grosbedou/PycharmProjects/NDORMS/data/matlab/slow_low_2.mat --python-utt /Users/grosbedou/PycharmProjects/NDORMS/results/notebook54_fix_strict_python_fascicle_prior_gate/strict_python_cumulative_calibrated_seed_2state_final_arrays.npz --python-timtrack /Users/grosbedou/PycharmProjects/NDORMS/results/notebook52_correct_video_fixed_emask_timtrack_gate/image_derived_timtrack_geofeatures_arrays.npz --video /Users/grosbedou/PycharmProjects/NDORMS/data/raw/UltraTimTrack_test.mp4 --utt-export /Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat --out-csv /Users/grosbedou/PycharmProjects/NDORMS/results/notebook54_fix_strict_python_fascicle_prior_gate/parity_metrics.csv


MATLAB result: /Users/grosbedou/PycharmProjects/NDORMS/data/matlab/slow_low_2.mat
Python final:  /Users/grosbedou/PycharmProjects/NDORMS/results/notebook54_fix_strict_python_fascicle_prior_gate/strict_python_cumulative_calibrated_seed_2state_final_arrays.npz
Python TimTrack-like: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook52_correct_video_fixed_emask_timtrack_gate/image_derived_timtrack_geofeatures_arrays.npz
image_depth_mm=50.7
image_height_px=562
image_width_px=706
mm_per_pixel=0.09021352
apox_1b=[20.0, 94.0, 168.0, 242.0, 316.0, 390.0, 464.0, 538.0, 612.0, 686.0]

comparison                                n       bias        mae       rmse     corr
-------------------------------------------------------------------------------------
final_FL_mm                            2666    -0.5399     1.4631     1.7663   0.9849
final_PEN_deg                          2666     0.2438     0.7204     0.8786   0.9842
final_ANG_deg                          2666     0.2935     0.7328    

,comparison,n,bias,mae,rmse,corr
0,final_FL_mm,2666,-0.539879,1.463055,1.766256,0.984923
1,final_PEN_deg,2666,0.243809,0.720394,0.878618,0.984237
2,final_ANG_deg,2666,0.293495,0.732827,0.886312,0.985927
3,timtrack_alpha_deg,2666,0.437734,0.837584,2.010149,0.933033
4,timtrack_phi_vs_python_pen_deg,2666,0.412744,0.891571,2.022837,0.926660
5,timtrack_formula_faslen_px,2666,-10.901087,19.975458,51.213782,0.910619
6,timtrack_gamma_deep_apo_deg,2666,0.049359,0.056801,0.090288,0.983819
7,timtrack_betha_super_apo_deg,2666,0.024990,0.110834,0.230326,0.848307
8,timtrack_super_pos_y1,2666,1.356007,1.607188,2.303977,0.821358
9,timtrack_super_pos_y2,2666,1.048207,1.185294,1.502483,0.885368


## Gate Decision

In [9]:
best_row = final_metrics_df.loc[final_metrics_df["variant"] == best_name].iloc[0].to_dict()
strict_row = final_metrics_df.loc[final_metrics_df["variant"] == "strict_one_step_python_timtrack_seed"].iloc[0].to_dict()
cumulative_row = final_metrics_df.loc[final_metrics_df["variant"] == "cumulative_python_timtrack_seed"].iloc[0].to_dict()

readiness = pd.DataFrame([
    {
        "gate": "strict_python_one_step_existing",
        "passes": bool(strict_row["passes_final_targets"]),
        "FL_RMSE_mm": strict_row["FL_RMSE_mm"],
        "PEN_RMSE_deg": strict_row["PEN_RMSE_deg"],
        "ANG_RMSE_deg": strict_row["ANG_RMSE_deg"],
        "note": "Current strict Python TimTrack per-frame prior from notebook 53.",
    },
    {
        "gate": "strict_python_cumulative_python_seed",
        "passes": bool(cumulative_row["passes_final_targets"]),
        "FL_RMSE_mm": cumulative_row["FL_RMSE_mm"],
        "PEN_RMSE_deg": cumulative_row["PEN_RMSE_deg"],
        "ANG_RMSE_deg": cumulative_row["ANG_RMSE_deg"],
        "note": "Uses packaged cumulative affine propagation from Python TimTrack frame-0 seed.",
    },
    {
        "gate": "strict_python_cumulative_calibrated_seed",
        "passes": bool(best_row["passes_final_targets"]),
        "FL_RMSE_mm": best_row["FL_RMSE_mm"],
        "PEN_RMSE_deg": best_row["PEN_RMSE_deg"],
        "ANG_RMSE_deg": best_row["ANG_RMSE_deg"],
        "note": f"Diagnostic ceiling with target-calibrated alpha0={best_alpha:.2f} deg; needs independent seed estimator before production use.",
    },
])
readiness.to_csv(OUT / "readiness.csv", index=False)

summary = {
    "notebook": "54_fix_strict_python_fascicle_prior_gate.ipynb",
    "new_package_helpers": [
        "ultrasound_tracker.matlab_timtrack.fascicle_segment_from_aponeuroses_and_alpha",
        "ultrasound_tracker.matlab_timtrack.fascicle_segment_from_geofeature(alpha_override=...)",
        "ultrasound_tracker.ultratrack_klt.propagate_cumulative_affines",
    ],
    "uses_python_aponeurosis_state": True,
    "uses_python_klt_affines": True,
    "uses_python_timtrack_alpha": True,
    "best_diagnostic_seed_alpha_deg": best_alpha,
    "best_diagnostic_final_npz": str(npz_path),
    "parity_metrics_csv": str(compare_csv),
    "strict_existing_passes": bool(strict_row["passes_final_targets"]),
    "cumulative_python_seed_passes": bool(cumulative_row["passes_final_targets"]),
    "calibrated_seed_passes": bool(best_row["passes_final_targets"]),
    "calibrated_seed_FL_RMSE_mm": float(best_row["FL_RMSE_mm"]),
    "calibrated_seed_PEN_RMSE_deg": float(best_row["PEN_RMSE_deg"]),
    "calibrated_seed_ANG_RMSE_deg": float(best_row["ANG_RMSE_deg"]),
    "remaining_independent_pipeline_blocker": "First-frame fascicle seed/angle estimator; calibrated seed is target-selected in this notebook.",
}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
readiness

,gate,passes,FL_RMSE_mm,PEN_RMSE_deg,ANG_RMSE_deg,note
0,strict_python_one_step_existing,False,3.953294,2.057106,2.067604,Current strict Python TimTrack per-frame prior...
1,strict_python_cumulative_python_seed,False,2.422119,1.312892,1.324520,Uses packaged cumulative affine propagation fr...
2,strict_python_cumulative_calibrated_seed,True,1.766256,0.878618,0.886312,Diagnostic ceiling with target-calibrated alph...
